# Somatotopic Decoding — Linear SVC
**Task:** S1Map | **ROI:** Left S1 (Destrieux G_postcentral)  
**Classes:** Middle_Finger · Hand · Forearm · Arm (4-way, imbalanced)  
**Cross-validation:** Leave-One-Run-Out (LORO, 4 folds)  
**Classifier:** LinearSVC (C=3.0, OVR, `class_weight='balanced'`)  
**Features:** ROI BOLD averaged in a ±1-volume window around the HRF peak (~6 s delay)


In [ ]:
# Core imports and global random seed
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


In [ ]:
# Electrode-to-region mapping (20 electrodes → 4 body regions)
REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand':          ['E4', 'E5', 'E6', 'E7'],
    'Forearm':       ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm':           ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20'],
}

ELECTRODE_TO_REGION = {
    electrode: region
    for region, electrodes in REGION_TO_ELECTRODES.items()
    for electrode in electrodes
}


In [ ]:
# BIDS dataset paths and experiment parameters
from pathlib import Path
import json

BIDS_ROOT = Path(r"../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
EVENT_DIR = BIDS_ROOT / "events"

session            = "ses-01"
task               = "task-S1Map"
space              = "MNI152NLin2009cAsym"
n_runs_per_subject = 4

HRF_DELAY = 6.0  # seconds from stimulus onset to HRF peak
WINDOW    = 1    # volumes to average on each side of the peak

In [ ]:
import re

derivative_subjects = sorted(d.name for d in DERIVATIVES.iterdir() if d.is_dir() and d.name.startswith("sub-"))

event_subjects = set()
for ev_path in EVENT_DIR.glob(f"sub-*_{session}_{task}_run-*_events.tsv"):
    m = re.match(r"^(sub-[^_]+)_", ev_path.name)
    if m:
        event_subjects.add(m.group(1))
event_subjects = sorted(event_subjects)

subjects = sorted(set(derivative_subjects) & set(event_subjects))

print(f"Subjects in derivatives: {len(derivative_subjects)}")
print(f"Subjects in events:      {len(event_subjects)}")
print(f"Subjects included:       {len(subjects)}")
print(subjects)

In [ ]:
# Read TR from the BIDS JSON sidecar of the first run
bold_json = (BIDS_ROOT / subjects[0] / session / "func" /
             f"{subjects[0]}_{session}_{task}_run-1_bold.json")
with open(bold_json, 'r', encoding='utf-8') as f:
    tr = float(json.load(f)["RepetitionTime"])

print(f"Subject: {subjects[0]}  |  Runs: {n_runs_per_subject}  |  TR: {tr} s")
print(f"HRF delay: {HRF_DELAY} s  |  Window: ±{WINDOW} vol")


In [ ]:
# Output directories and analysis log
RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_LOG = RESULTS_DIR / "analysis_log.txt"
log_file = open(OUTPUT_LOG, 'w', encoding='utf-8')


## Load Events

In [ ]:
# Load events TSV files and filter to stimulation trials only
import pandas as pd

subject    = subjects[0]
all_events = []
for run in range(1, n_runs_per_subject + 1):
    events_path = (BIDS_ROOT / subject / session / "func" /
                   f"{subject}_{session}_{task}_run-{run}_events.tsv")
    df = pd.read_csv(events_path, sep='\t')
    df['subject'] = subject
    df['run'] = run
    all_events.append(df)

events_df   = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)

print(f"Total events: {len(events_df)}  |  Stimulation events: {len(stim_events)}")
print(f"Samples per class: {stim_events['region'].value_counts().sort_index().to_dict()}")


In [ ]:
# Load first BOLD run as reference image and build S1 ROI mask from Destrieux atlas
from nilearn.image import load_img, index_img, resample_to_img, new_img_like
from nilearn.datasets import fetch_atlas_destrieux_2009

first_run_path = (DERIVATIVES / subjects[0] / session / "func" /
                  f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii")
first_run_img = load_img(str(first_run_path))
ref_img       = index_img(first_run_img, 0)
print(f"BOLD shape: {first_run_img.shape}")

atlas      = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img  = load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()

mask_data         = np.isin(atlas_data, s1_indices).astype('uint8')
s1_mask           = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')

print(f"S1 voxels (atlas space): {int(np.sum(mask_data))}")
print(f"S1 voxels (BOLD space):  {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")


In [ ]:
# Fit the NiftiMasker on the S1 ROI and save a visualization of the mask
from nilearn.maskers import NiftiMasker
from nilearn import plotting

masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)

print('Selected atlas regions:')
for i in s1_indices:
    print(f"  [{i}] {atlas.labels[i]}")

display = plotting.plot_roi(s1_mask_resampled, bg_img=ref_img,
                            title='S1 ROI Mask (Left G_postcentral)',
                            display_mode='ortho', cut_coords=(0, -20, 60))
display.savefig(FIGURES_DIR / 's1_mask.png', dpi=150)
display.close()
print(f"\nMask saved → {FIGURES_DIR / 's1_mask.png'}")


In [ ]:
# Extract BOLD features: for each trial, average volumes in the ±WINDOW HRF window
from nilearn.image import mean_img

feature_rows, labels, groups = [], [], []

for run in range(1, n_runs_per_subject + 1):
    func_path = (DERIVATIVES / subject / session / "func" /
                 f"{subject}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii")
    img        = load_img(str(func_path))
    run_length = img.shape[3]

    run_events = stim_events[
        (stim_events['subject'] == subject) &
        (stim_events['run'] == run)
    ].sort_values('onset')

    for _, ev in run_events.iterrows():
        peak_vol = int(np.round((float(ev['onset']) + HRF_DELAY) / tr))
        if peak_vol >= run_length:
            continue
        vols     = list(range(max(0, peak_vol - WINDOW), min(run_length, peak_vol + WINDOW + 1)))
        averaged = mean_img(index_img(img, vols), copy_header=True)
        feature_rows.append(masker.transform(averaged).ravel())
        labels.append(str(ev['region']))
        groups.append(run)


In [ ]:
# Stack into feature matrix and print dataset summary
X_features = np.vstack(feature_rows)
y          = np.asarray(labels)
groups     = np.asarray(groups)

print(f"Feature matrix : {X_features.shape}  (samples × voxels)")
print(f"Classes        : {np.unique(y)}")
print(f"Samples/class  : { {c: int(np.sum(y == c)) for c in np.unique(y)} }")


In [ ]:
# Leave-One-Run-Out cross-validation setup
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut

logo    = LeaveOneGroupOut()
n_folds = logo.get_n_splits(groups=groups)

samples_train = len(y) * (n_folds - 1) / n_folds
samples_test  = len(y) / n_folds

print(f"CV strategy : Leave-One-Run-Out ({n_folds} folds)")
print(f"Train/fold  : {samples_train:.0f} samples  ({samples_train / len(np.unique(y)):.1f} per class)")
print(f"Test/fold   : {samples_test:.0f} samples   ({samples_test  / len(np.unique(y)):.1f} per class)")


## Baseline SVC — LORO Cross-Validation

Can individual body regions be decoded from S1 BOLD activity patterns  
using a Linear SVM with leave-one-run-out cross-validation?


In [ ]:
# Define LinearSVC pipeline: z-score normalization → balanced-weight SVM (OVR)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

clf  = LinearSVC(C=3.0, class_weight='balanced', dual='auto',
                 max_iter=20000, random_state=RANDOM_SEED)
pipe = Pipeline(steps=[('scale', StandardScaler()), ('clf', clf)])

all_classes  = np.unique(y)
chance_level = 1.0 / len(all_classes)
print(f"Classes ({len(all_classes)}): {list(all_classes)}")
print(f"Chance level: {chance_level:.4f} ({chance_level*100:.1f}%)")


In [ ]:
# LORO cross-validation: train on 3 runs, test on the left-out run
y_pred    = np.empty_like(y, dtype=object)
fold_acc  = []
fold_bacc = []

for fold_i, (train_idx, test_idx) in enumerate(logo.split(X_features, y, groups), 1):
    left_out = int(np.unique(groups[test_idx])[0])
    pipe.fit(X_features[train_idx], y[train_idx])
    y_pred[test_idx] = pipe.predict(X_features[test_idx])

    acc  = accuracy_score(y[test_idx], y_pred[test_idx])
    bacc = balanced_accuracy_score(y[test_idx], y_pred[test_idx])
    fold_acc.append(acc)
    fold_bacc.append(bacc)

    y_true_fold = y[test_idx]
    y_pred_fold = y_pred[test_idx]
    n_correct   = int(np.sum(y_true_fold == y_pred_fold))
    n_total     = len(y_true_fold)

    print(f"\nFold {fold_i} (run {left_out}):  {n_correct}/{n_total} correct")
    print(f"  Accuracy: {acc:.4f}   Balanced accuracy: {bacc:.4f}")
    print(f"  True:      {list(y_true_fold)}")
    print(f"  Predicted: {list(y_pred_fold)}")
    print(f"  Match:     {[t == p for t, p in zip(y_true_fold, y_pred_fold)]}")


In [ ]:
# Aggregate per-fold metrics and report final performance
fold_acc  = np.asarray(fold_acc)
fold_bacc = np.asarray(fold_bacc)

mean_accuracy = float(fold_acc.mean())
std_accuracy  = float(fold_acc.std())
mean_bacc     = float(fold_bacc.mean())
std_bacc      = float(fold_bacc.std())

print(f"Mean accuracy:          {mean_accuracy:.4f} ± {std_accuracy:.4f}")
print(f"Mean balanced accuracy: {mean_bacc:.4f} ± {std_bacc:.4f}")
print(f"Chance level:           {chance_level:.4f} ({chance_level*100:.1f}%)")
print(f"Above chance:           {(mean_bacc - chance_level) / chance_level * 100:+.1f}%")


In [ ]:
# Bar chart: per-fold accuracy and balanced accuracy vs chance level
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
x     = np.arange(1, len(fold_acc) + 1)
width = 0.38

ax.bar(x - width / 2, fold_acc,  width=width, color='#1f77b4', alpha=0.85,
       edgecolor='black', label='Accuracy')
ax.bar(x + width / 2, fold_bacc, width=width, color='#ff7f0e', alpha=0.85,
       edgecolor='black', label='Balanced accuracy')
ax.axhline(chance_level, linestyle=':', linewidth=2, color='gray',
           label=f'Chance = {chance_level:.3f}')
ax.axhline(mean_bacc, linestyle='--', linewidth=2, color='red', alpha=0.5,
           label=f'Mean bacc = {mean_bacc:.3f}')

ax.set_xlabel('Fold (left-out run)', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('4-Way Region Classification (sub-p0002)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_ylim(0, max(0.40, max(fold_bacc) * 1.15))
ax.grid(axis='y', alpha=0.3)
ax.legend(loc='upper right')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '4way_performance.png', dpi=300, bbox_inches='tight')
plt.show()


## Confusion Matrix

In [ ]:
# Confusion matrices: raw counts and row-normalized (per-class recall)
from sklearn.metrics import confusion_matrix

cm      = confusion_matrix(y, y_pred, labels=all_classes)
row_sum = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, np.where(row_sum == 0, 1, row_sum))

per_class_acc = np.diag(cm_norm)
for cls, acc in zip(all_classes, per_class_acc):
    print(f"  {cls:<15}: {acc:.3f}")
print(f"\nMean recall: {per_class_acc.mean():.3f} ± {per_class_acc.std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(cm, cmap='Blues', aspect='auto')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(cm_norm, cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

for i in range(len(all_classes)):
    val   = float(cm_norm[i, i])
    color = 'white' if val > 0.5 else 'black'
    axes[1].text(i, i, f"{val:.2f}", ha='center', va='center', fontsize=8, color=color)

for ax in axes:
    ax.set_xticks(np.arange(len(all_classes)))
    ax.set_yticks(np.arange(len(all_classes)))
    ax.set_xticklabels(all_classes, rotation=90)
    ax.set_yticklabels(all_classes)
    ax.set_xlabel('Predicted Region')
    ax.set_ylabel('True Region')

plt.tight_layout()
fig.savefig(FIGURES_DIR / '4way_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Refit baseline SVC on all data and extract per-class decision weights
from sklearn.base import clone

full_pipe = Pipeline(steps=[('scale', StandardScaler()), ('clf', clone(clf))])
full_pipe.fit(X_features, y)

scaler       = full_pipe.named_steps['scale']
svm          = full_pipe.named_steps['clf']
weight_voxel = svm.coef_ / scaler.scale_[np.newaxis, :]

print(f'Weight shape: {weight_voxel.shape}  (n_classes={len(all_classes)}, n_voxels)')


## Weight Mapping

Project SVM decision weights back to voxel space and visualize per-class activation patterns.


In [ ]:
# Per-class absolute weight maps projected back to brain volume (top-25% voxels highlighted)
from IPython.display import display

for c in range(len(all_classes)):
    weight_img = masker.inverse_transform(np.abs(weight_voxel[c]).reshape(1, -1))
    view = plotting.view_img(
        weight_img,
        title=f'SVM Weights: {all_classes[c]}',
        threshold=np.percentile(np.abs(weight_voxel[c]), 75),
        colorbar=True,
        symmetric_cmap=False,
    )
    display(view)


In [ ]:
# Winner-take-all somatotopic map: each voxel is assigned to its dominant class
TOP_PERCENTILE = 50  # keep only the top-50% most strongly weighted voxels

abs_weights   = np.abs(weight_voxel)
winner_map    = np.argmax(abs_weights, axis=0)
max_weight    = np.amax(abs_weights, axis=0)
threshold     = np.percentile(max_weight, TOP_PERCENTILE)
winner_values = np.where(max_weight >= threshold, winner_map + 1, 0).astype(np.float64)
winner_img    = masker.inverse_transform(winner_values.reshape(1, -1))

somatotopic_title = ("Somatotopic Map (" +
                     ", ".join(f"{i+1}={all_classes[i]}" for i in range(len(all_classes))) +
                     ")")

view = plotting.view_img_on_surf(
    winner_img,
    surf_mesh='fsaverage',
    title=somatotopic_title,
    symmetric_cmap=False,
    vmin=1,
    vmax=len(all_classes),
    threshold=1,
    vol_to_surf_kwargs={"interpolation": "nearest_most_frequent"},
)
view
